# 🎮 SheepBell - Google Colab 起動ノートブック
ゲームLQA用 自動動画クリップ & 文字起こしツール (Gradio UI)

### 📌 全体の流れ
1. **ランタイムの設定**: メニューの **「ランタイム」 > 「ランタイムのタイプを変更」** で **T4 GPU** が選択されていることを確認します。
2. **セル 1〜3 を順番に実行**: 発行された `https://xxxx.gradio.live` のリンクを開いて動画を処理します。
3. **処理後の便利機能 (セル 4, 5)**: スプレッドシートへの直接変換や、生成物の一括Zipダウンロードが可能です。

### 1. (任意) Google Drive のマウント
OBSで録画した大容量動画を Google Drive 経由で直接読み書きする場合は実行してください。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. リポジトリのクローン / 最新化 (初回はclone、2回目以降はpull)
リポジトリが未取得ならcloneし、既にあればpullで最新状態に更新します。

In [ ]:
import os

# 自身のGitHubリポジトリURLに変更してください
REPO_URL = "https://github.com/your-username/SheepBell.git"
TARGET_DIR = "/content/SheepBell"

if not os.path.exists(TARGET_DIR):
    print("🚀 リポジトリをクローン中...")
    !git clone {REPO_URL} {TARGET_DIR}
else:
    print("🔄 既存リポジトリを最新に更新 (git pull) 中...")
    %cd {TARGET_DIR}
    !git pull

%cd {TARGET_DIR}

# 依存パッケージのインストール (GPUライブラリ含む)
!pip install -q gradio faster-whisper silero-vad onnxruntime-gpu imageio-ffmpeg gspread nvidia-cublas-cu12 nvidia-cudnn-cu12

# CUDA ライブラリパスの設定
import os
os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":/usr/local/lib/python3.10/dist-packages/nvidia/cublas/lib:/usr/local/lib/python3.11/dist-packages/nvidia/cublas/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cublas/lib"

### 3. Web UI の起動 (Gradio Share URL 発行)
実行すると一番下に `Running on public URL: https://xxxx.gradio.live` と表示されるので、そのリンクをクリックしてブラウザで利用します。
*(※UIの利用中は、このセルを実行中のままにしておきます)*

In [ ]:
# GPU上でGradio UIを起動し、公開URLを発行します
!python app.py --share

---
## 🎁 処理完了後の便利ツール (後処理)
Web UIでの処理が終わったら、以下のセルを活用できます。

### 4. 📊 生成された CSV を Google スプレッドシートに一発変換して開く
実行すると Google アカウントの認証ポップアップが出ます。許可すると、マイドライブにスプレッドシートが新規作成されて URL が発行されます。

In [ ]:
import csv
import datetime
from google.colab import auth
from google.auth import default
import gspread

# 1. 認証
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 2. CSV の読み込み (出力先ディレクトリに合わせて変更可能)
csv_path = "./issues/lqa_issues.csv"

with open(csv_path, "r", encoding="utf-8-sig") as f:
    rows = list(csv.reader(f))

# 3. 日時付きのスプレッドシートを作成して書き込み
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
sheet_title = f"LQA_Issues_{timestamp}"
sh = gc.create(sheet_title)
worksheet = sh.get_worksheet(0)
worksheet.update(rows)

print(f"🎉 Google スプレッドシートを作成しました！: {sheet_title}")
print(f"🔗 開く -> {sh.url}")

### 5. 📦 生成ファイル (動画・画像・CSV) を Zip で一括ダウンロード
Colabの一時領域（`./issues`）に出力したファイルをPCにまとめて保存したい場合に実行してください。

In [ ]:
# issues フォルダを zip に圧縮してPCにダウンロード
!zip -r /content/issues.zip ./issues
from google.colab import files
files.download("/content/issues.zip")